# Kev 800M: best.pt of every paper run on Grummon 2025, Kreps 2020 (COVID), and Kantorowicz 2026

What and why: `README.md` in this folder.

In [1]:
import json
import os
import sys
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

# Walk up from the notebook folder to locate the repository's `code/` root
CODE_DIR = next(
    (c for p in (Path.cwd(), *Path.cwd().parents) for c in (p, p / "code") if all((c / d).is_dir() for d in ("kev", "data", "training"))),
    None,
)
if CODE_DIR is None:
    raise RuntimeError(f"No code/ root containing kev/, data/, and training/ at or above {Path.cwd()}")

BETA_TRAINING_DIR = CODE_DIR / "train-notebooks" / "beta-training"
PAPER_DIR = BETA_TRAINING_DIR / "ms_paper_beta_loss"

# Run from `code/` so data loaders and shared disk caches resolve identically to the training notebooks
os.chdir(CODE_DIR)
for _p in (str(CODE_DIR), str(CODE_DIR / "kev"), str(BETA_TRAINING_DIR), str(PAPER_DIR)):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from models.kev_pointer import KevPointerModel
from training import free_gpu

import beta_ols
from beta_train import runs as run_store
from beta_train.model_state import reset_trainable, set_checkpointing, snapshot_trainable
from beta_train.progress import PROGRESS_INTERVAL_SEC, HtmlProgress
from beta_train.validation import PopulationValidator, validate_all, validate_all_cached, validation_metrics_frame

NOTEBOOK = "04-test-best-800m"
CACHE_DIR = CODE_DIR / ".cache" / "beta_training"
MODEL_SLUG = "kev-0.8b"
# Runs pulled from Rorqual by pull-rorqual-runs.sh (run files + checkpoints/best.pt)
RUNS_DIR = PAPER_DIR / "runs-rorqual" / MODEL_SLUG
# Each run's test results go in <run folder>/test_best/
TEST_FOLDER_NAME = "test_best"
ROW1_DIR = RUNS_DIR / "row1_base_untrained"

print(f"RUNS_DIR: {RUNS_DIR}")


RUNS_DIR: /home/dw/github/decision-models-finetuning/code/train-notebooks/beta-training/ms_paper_beta_loss/runs-rorqual/kev-0.8b


In [2]:
# Same backbone and LoRA shape as the training notebooks, so best.pt loads into it
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required")
device = "cuda"
free_gpu()

KEV_BASE = "jaredpalmer/kev-0.8b@bf75a6a8848ea6960ff2ed108d9ed44c2941174f"
LORA_RANK = 64
WEIGHTS_DTYPE = "bf16"
HEAD_TEMPERATURE = 1.0

kev_model = KevPointerModel.build(
    KEV_BASE,
    device=device,
    lora_rank=LORA_RANK,
    weights_dtype=WEIGHTS_DTYPE,
)
kev_model.net.head.temperature = HEAD_TEMPERATURE
set_checkpointing(kev_model, False)

INIT_TRAINABLE_STATE = snapshot_trainable(kev_model)
# Same key as the training notebooks, so their cached step-0 validations are reused
MODEL_KEY = {
    "base": KEV_BASE,
    "lora_rank": LORA_RANK,
    "weights_dtype": WEIGHTS_DTYPE,
    "head_temperature": HEAD_TEMPERATURE,
}
print(f"{KEV_BASE}: {len(INIT_TRAINABLE_STATE)} trainable tensors")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

jaredpalmer/kev-0.8b@bf75a6a8848ea6960ff2ed108d9ed44c2941174f: 376 trainable tensors


## Test datasets

In [3]:
TEST_DATASETS = ["grummon2025", "covid", "kantorowicz2026"]
VALIDATION_MAX_QUESTIONS = 1024
VALIDATION_MAX_CHARS = 1_000_000

validators = {}
for dataset_name in TEST_DATASETS:
    dataset_module = beta_ols.DATASET_MODULES[dataset_name]
    choice_df = dataset_module.load_choice_df()
    rows = dataset_module.regression_rows(choice_df)
    specs = dataset_module.regression_specs(rows)
    validators[dataset_name] = PopulationValidator(
        dataset_module,
        choice_df,
        specs,
        max_questions=VALIDATION_MAX_QUESTIONS,
        max_chars=VALIDATION_MAX_CHARS,
    )
display(pd.DataFrame([v.describe() for v in validators.values()]).set_index("validator"))


# Writes metrics.csv (one row per test dataset), betas.csv (model vs human betas, stacked) and test.json (what was tested)
def save_test_results(test_dir: Path, validation: dict, test_info: dict) -> None:
    test_dir.mkdir(parents=True, exist_ok=True)
    metrics_df = validation_metrics_frame(validation).T
    metrics_df.index.name = "dataset"
    run_store.write_text_atomic(test_dir / "metrics.csv", metrics_df.to_csv())
    run_store.write_betas_file(test_dir / "betas.csv", validation)
    run_store.write_text_atomic(test_dir / "test.json", json.dumps(test_info, indent=2))


# True when test_dir already holds results for this checkpoint step, model config, and test datasets
def already_tested(test_dir: Path, checkpoint_step: int) -> bool:
    test_json = test_dir / "test.json"
    if not test_json.exists():
        return False
    stored = json.loads(test_json.read_text())
    return (
        stored["checkpoint_step"] == checkpoint_step
        and stored["model"] == MODEL_KEY
        and stored["test_datasets"] == TEST_DATASETS
    )


# Loads already-saved dataset results when test_dir matches this checkpoint step and model config,
# so adding a new dataset to TEST_DATASETS evaluates only the missing dataset(s) and merges cleanly.
def load_existing_test_results(
    test_dir: Path, checkpoint_step: int
) -> dict[str, tuple[dict[str, float], pd.DataFrame]]:
    test_json = test_dir / "test.json"
    if not test_json.exists():
        return {}
    metrics_csv = test_dir / "metrics.csv"
    betas_csv = test_dir / "betas.csv"
    if not metrics_csv.exists() or not betas_csv.exists():
        raise FileNotFoundError(f"Incomplete test_best directory in {test_dir}")
    stored = json.loads(test_json.read_text())
    if stored["checkpoint_step"] != checkpoint_step or stored["model"] != MODEL_KEY:
        return {}
    stored_datasets = list(stored["test_datasets"])
    metrics_df = pd.read_csv(metrics_csv, float_precision="round_trip").set_index("dataset")
    beta_dfs = run_store.read_betas_file(betas_csv)
    if list(metrics_df.index) != stored_datasets or list(beta_dfs.keys()) != stored_datasets:
        raise ValueError(
            f"Inconsistent test_best artifacts in {test_dir}: "
            f"test.json={stored_datasets}, metrics.csv={list(metrics_df.index)}, betas.csv={list(beta_dfs.keys())}"
        )
    return {
        ds: ({k: float(v) for k, v in metrics_df.loc[ds].items()}, beta_dfs[ds])
        for ds in stored_datasets
        if ds in TEST_DATASETS
    }


,kind,tasks,packed_persona_states,regressions,max_questions,max_chars,graph_limits,prep_sec
validator,,,,,,,,
grummon2025,fused,33798,4906,grummon2025,1024,1000000,"BANK_WIDTH=512, GRAPH_STATE=512, GRAPH_STATES=...",6.482
covid,fused,9855,1845,covid,1024,1000000,"BANK_WIDTH=512, GRAPH_STATE=512, GRAPH_STATES=...",1.937
kantorowicz2026,fused,26823,42,"kantorowicz2026/computers, kantorowicz2026/lau...",1024,1000000,"BANK_WIDTH=512, GRAPH_STATE=512, GRAPH_STATES=...",2.847


## Row 1: untrained model

In [ ]:
# Untrained model (step 0): reuse existing test_best/ results when step and model match,
# and evaluate only missing datasets (cached in code/.cache/beta_training)
row1_test_dir = ROW1_DIR / TEST_FOLDER_NAME
if already_tested(row1_test_dir, 0):
    step0_validation = load_existing_test_results(row1_test_dir, 0)
    print("Row 1 untrained (step 0): already tested, loaded from test_best/")
else:
    existing_step0 = load_existing_test_results(row1_test_dir, 0)
    missing_datasets = [d for d in TEST_DATASETS if d not in existing_step0]
    if missing_datasets:
        missing_validators = {d: validators[d] for d in missing_datasets}
        new_step0, step0_cache_report = validate_all_cached(
            kev_model,
            missing_validators,
            INIT_TRAINABLE_STATE,
            model_key=MODEL_KEY,
            cache_dir=CACHE_DIR,
        )
        display(step0_cache_report)
    else:
        new_step0 = {}
    step0_validation = {
        d: (new_step0[d] if d in new_step0 else existing_step0[d])
        for d in TEST_DATASETS
    }
    step0_info = {
        "notebook": NOTEBOOK,
        "model": MODEL_KEY,
        "row": "Row 1",
        "checkpoint": "untrained",
        "checkpoint_step": 0,
        "test_datasets": TEST_DATASETS,
    }
    save_test_results(row1_test_dir, step0_validation, step0_info)
    reused = [d for d in TEST_DATASETS if d not in missing_datasets]
    note = f"tested {missing_datasets}, reused {reused}" if reused else f"tested {missing_datasets}"
    print(f"Row 1 untrained (step 0): {note}")
display(validation_metrics_frame(step0_validation).round(5))


## best.pt of every finished run

In [5]:
# best.pt of every finished run in RUNS_DIR; unfinished runs are skipped (rerun this notebook once they finish)
progress_bar = HtmlProgress(
    total=0, initial=0, description="test (waiting)", unit="batch", min_interval_sec=PROGRESS_INTERVAL_SEC
)
for run_dir in sorted(RUNS_DIR.iterdir()):
    if not (run_dir / "run.json").exists():
        continue
    run_info = run_store.read_run_json(run_dir)
    row_code = run_info["identity"]["run_metadata"]["paper_row"]
    if run_info["status"] != "finished":
        print(f"{row_code} {run_dir.name}: {run_info['status']}, skipped")
        continue
    test_dir = run_dir / TEST_FOLDER_NAME
    checkpoint = run_store.load_best(run_dir)
    best = checkpoint["best"]
    checkpoint_step = int(best["step"])
    if already_tested(test_dir, checkpoint_step):
        print(f"{row_code} {run_dir.name}: best.pt (step {checkpoint_step}) already tested, skipped")
        continue
    existing = load_existing_test_results(test_dir, checkpoint_step)
    missing_datasets = [d for d in TEST_DATASETS if d not in existing]
    if missing_datasets:
        reset_trainable(kev_model, checkpoint["trainable"])
        missing_validators = {d: validators[d] for d in missing_datasets}
        new_validation = validate_all(kev_model, missing_validators, progress_bar)
    else:
        new_validation = {}
    validation = {
        d: (new_validation[d] if d in new_validation else existing[d])
        for d in TEST_DATASETS
    }
    test_info = {
        "notebook": NOTEBOOK,
        "model": MODEL_KEY,
        "row": row_code,
        "checkpoint": "best.pt",
        "checkpoint_step": checkpoint_step,
        "best_metric": best["metric"],
        "best_value": float(best["value"]),
        "test_datasets": TEST_DATASETS,
    }
    save_test_results(test_dir, validation, test_info)
    reused = [d for d in TEST_DATASETS if d not in missing_datasets]
    note = f"tested {missing_datasets}, reused {reused}" if reused else f"tested {missing_datasets}"
    print(f"{row_code} {run_dir.name}: best.pt (step {checkpoint_step}) {note}")
    display(validation_metrics_frame(validation).round(5))
progress_bar.close()
reset_trainable(kev_model, INIT_TRAINABLE_STATE)


Row 2 (Ablation) row2_ablation_wu_table_no_kl: best.pt (step 576) tested


,grummon2025,covid
kl_loss,1.02090,1.17363
beta_mse,0.02690,0.02046
beta_mae,0.16078,0.09866
beta_mape_pct,156.75805,233.67983
spearman_rho,0.70000,0.66364
eval_prep_sec,0.03094,0.00843
eval_gpu_sec,60.61859,27.05923
eval_wall_sec,62.38241,28.67500
eval_ms_per_task,1.84574,2.90969


Row 2 row2_wu_table_anchored: best.pt (step 864) tested


,grummon2025,covid
kl_loss,0.96538,1.14068
beta_mse,0.00731,0.01335
beta_mae,0.08538,0.08946
beta_mape_pct,82.30312,193.66948
spearman_rho,0.70000,0.69091
eval_prep_sec,0.02610,0.00879
eval_gpu_sec,60.67761,27.08822
eval_wall_sec,62.38624,28.85716
eval_ms_per_task,1.84586,2.92818


Row 2b row2b_wu_ce_oracle: best.pt (step 640) tested


,grummon2025,covid
kl_loss,0.91534,1.08210
beta_mse,0.00353,0.00373
beta_mae,0.05163,0.05044
beta_mape_pct,40.33992,108.52474
spearman_rho,0.50000,0.79091
eval_prep_sec,0.03021,0.00942
eval_gpu_sec,60.74558,27.03404
eval_wall_sec,62.51586,28.62889
eval_ms_per_task,1.84969,2.90501


... (4 outputs left out)

Row 4 (Ablation) row4_hybrid_berke_ce_wu_table_no_kl: best.pt (step 2240) tested


,grummon2025,covid
kl_loss,0.87411,1.08414
beta_mse,0.00425,0.00463
beta_mae,0.04941,0.05803
beta_mape_pct,50.45800,129.45590
spearman_rho,0.60000,0.63636
eval_prep_sec,0.02751,0.01033
eval_gpu_sec,60.74998,27.09469
eval_wall_sec,62.63950,28.98938
eval_ms_per_task,1.85335,2.94159


Row 5 row5_full_berke_wu_ce_oracle: best.pt (step 1600) tested


,grummon2025,covid
kl_loss,0.87928,1.06686
beta_mse,0.00230,0.00256
beta_mae,0.03735,0.03936
beta_mape_pct,34.33993,84.99044
spearman_rho,0.50000,0.82727
eval_prep_sec,0.02868,0.01010
eval_gpu_sec,60.76075,27.13302
eval_wall_sec,62.87645,29.12337
eval_ms_per_task,1.86036,2.95519


## Summary

In [6]:
# Every tested checkpoint of this model: one row per (row, run folder, test dataset)
summary_frames = []
for test_json in sorted(RUNS_DIR.glob(f"*/{TEST_FOLDER_NAME}/test.json")):
    test_dir = test_json.parent
    test_info = json.loads(test_json.read_text())
    metrics_df = pd.read_csv(test_dir / "metrics.csv")
    metrics_df.insert(0, "checkpoint_step", test_info["checkpoint_step"])
    metrics_df.insert(0, "slug", test_dir.parent.name)
    metrics_df.insert(0, "row", test_info["row"])
    summary_frames.append(metrics_df)
summary_df = pd.concat(summary_frames, ignore_index=True)
summary_df = summary_df.sort_values(["dataset", "row", "slug"]).set_index(["dataset", "row", "slug"])
display(summary_df[["checkpoint_step", "beta_mse", "beta_mae", "spearman_rho", "kl_loss"]].round(5))


checkpoint_step  beta_mse  beta_mae  spearman_rho  kl_loss
dataset     row              slug                                                                                              
covid       Row 1            row1_base_untrained                                   0   0.01207   0.08735       0.55455  1.18456
            Row 2            row2_wu_table_anchored                              864   0.01335   0.08946       0.69091  1.14068
            Row 2 (Ablation) row2_ablation_wu_table_no_kl                        576   0.02046   0.09866       0.66364  1.17363
            Row 2b           row2b_wu_ce_oracle                                  640   0.00373   0.05044       0.79091  1.08210
            Row 3            row3_berke_ce_only                                 1152   0.00408   0.05298       0.70909  1.10543
            Row 4            row4_hybrid_berke_ce_wu_table_anchored             2240   0.00442   0.05586       0.63636  1.09213
            Row 4 (Ablation) row4_hybrid_berke_ce_wu_table_no_kl                2240   0.00463   0.05803       0.63636  1.08414
            Row 5            row5_full_berke_wu_ce_oracle                       1600   0.00256   0.03936       0.82727  1.06686
grummon2025 Row 1            row1_base_untrained                                   0   0.00546   0.06510       0.50000  1.05890
            Row 2            row2_wu_table_anchored                              864   0.00731   0.08538       0.70000  0.96538
            Row 2 (Ablation) row2_ablation_wu_table_no_kl                        576   0.02690   0.16078       0.70000  1.02090
            Row 2b           row2b_wu_ce_oracle                                  640   0.00353   0.05163       0.50000  0.91534
            Row 3            row3_berke_ce_only                                 1152   0.00325   0.05029       0.50000  0.95350
            Row 4            row4_hybrid_berke_ce_wu_table_anchored             2240   0.00355   0.04600       0.60000  0.91100
            Row 4 (Ablation) row4_hybrid_berke_ce_wu_table_no_kl                2240   0.00425   0.04941       0.60000  0.87411
            Row 5            row5_full_berke_wu_ce_oracle                       1600   0.00230   0.03735       0.50000  0.87928